# Validation — `kappa-lora-spectral-targeting`

**What this measures:** The repo's own MetaMathQA harness (the exact `run.py --verbose --clean <experiment>` → `temporary_results/*.json` pattern the requester's Colab demonstrates) emits `num_trainable_params`, which directly observes this PR's mechanism: κ-ranked top-50% spectral selection shrinking the adapted pool from 56 modules (published LoRA row: 9,175,040 params) to exactly 28, i.e. the "halves trainable params" claim as a computed structural bound (≤5,505,024); `test_accuracy` from the same run is now the sole guardrail (0.4705 floor), with forgetting/time/memory demoted to cost references per the reviewer so a noisy forgetting number cannot veto the parameter-count/accuracy question.

**How I read the claim:** The PR claims κ-LoRA spectral targeting: rank the base-weight matrices matched by target_modules by condition number, adapt only the top `condition_number_top_fraction` (paper: 0.5), and you cut trainable parameters roughly in half with no accuracy loss. We will validate this the way this repo validates every contribution: a new config `experiments/lora/llama-3.2-3B-rank32-kappa05/` (same r=32, alpha=64, dropout=0, target_modules=[q_proj, v_proj] as the published row, plus `condition_number_top_fraction: 0.5`) run through `method_comparison/MetaMathQA/run.py` and compared to the published `lora--llama-3.2-3B-rank32` row. The measured targets are `num_trainable_params` (must be consistent with exactly 28 of the 56 selected modules: ≤5,505,024 hard bound, ≈4,587,520 = exactly half at a 14/14 q/v split) and `test_accuracy` ≥ 0.4705 (row 0.4905 − the repo's 0.02 parity band) over 3 seeds, with time/memory/forgetting as no-regression cost checks. Support = parameter count in [3,670,016, 5,505,024] near the 50% point and accuracy inside the band on every seed. The caveat: this protocol's candidate pool is only q_proj/v_proj of two different shapes, so the paper's 'top 50% of matrices halves parameters' is a module-count claim that lands anywhere in 40–60% of baseline parameters depending on which type the condition numbers favor, and the paper's −16.2% time / −4.5% memory deltas come from a different setup and are reported, not asserted.

- ⚠️ Candidate pool: the paper ranks all weight matrices of the model; this protocol's comparator fixes target_modules=[q_proj, v_proj], so selection happens within 56 attention modules of two shapes — the cross-module-type (attn vs MLP) spectral selection the paper describes is not exercised.
- ⚠️ 'Halves params' is a module-count claim: with q_proj 1.5× larger than v_proj, top-28 selection yields 40–60% of baseline parameters (3,670,016–5,505,024), exactly 50% (4,587,520) only at a 14/14 split. The test must convert the claim to a computed parameter threshold, not assume 50%.
- ⚠️ The paper's −16.2% time and −4.5% memory were averaged over its own benchmark suite; on this harness the LoRA share of compute/memory is small (halving ~4.59M trainable params saves only ~37 MB of Adam state vs a 22.3 GB peak), so time/memory deltas may be inside run-to-run noise — they are cost/no-regression checks, not parity targets.

**Target metric:** `num_trainable_params`

**Repository:** [mayorquinmachines/peft](https://github.com/mayorquinmachines/peft) at commit [`51d77ab0c6ca`](https://github.com/mayorquinmachines/peft/commit/51d77ab0c6cac5bfec09637def630d6d9559f9fb)

**Benchmark:** the repository's own `method_comparison/MetaMathQA/run.py` over `experiments/lora/llama-3.2-3B-rank32-kappa05` — not a synthesized stand-in, so the numbers are comparable to what this repository publishes.

**Nothing here has been executed** — there are no outputs and no result is being claimed. Review the measurement, edit the configuration or criteria if it is wrong, then mention `@remyx validate` to run it on Remyx compute — or run the cells top to bottom yourself on a machine with a GPU.

In [1]:
# Parameters (Remyx passes the commit it measures as `ref`)
variant = "feature"
ref = ""
seed = 0

In [2]:
# Parameters
variant = "feature"
ref = "627b015c7fd1ae77e5110980be98217760f6479f"
seed = 0


## 1. Environment

A CUDA GPU is required; the published protocol peaks above 22 GB.

In [3]:
!nvidia-smi -L
import sys, torch
print(f"python {sys.version.split()[0]} · torch {torch.__version__} · cuda {torch.cuda.is_available()}")

GPU 0: NVIDIA L4 (UUID: GPU-00052cb1-9e9f-6b87-43ba-44e1ff241737)


python 3.12.3 · torch 2.14.0+cu126 · cuda True


## 2. The code under test

Clone the repository and check out exactly the commit that was validated, then install it in editable mode so the harness imports this checkout. When this notebook runs on Remyx compute the checkout already exists at that commit, and this cell only confirms it.

In [4]:
import os, subprocess, sys
REPO_URL = "https://github.com/mayorquinmachines/peft"
COMMIT = ref or "51d77ab0c6cac5bfec09637def630d6d9559f9fb"

def _sh(*cmd):
    return subprocess.run(cmd, check=True, text=True, capture_output=True).stdout.strip()

def _at_commit():
    try:
        return os.path.isdir(".git") and _sh("git", "rev-parse", "HEAD").startswith(COMMIT)
    except Exception:
        return False

if not _at_commit():
    if not os.path.isdir("repo"):
        _sh("git", "clone", "--quiet", REPO_URL, "repo")
    os.chdir("repo")
    _sh("git", "fetch", "--quiet", "--depth=1", "origin", COMMIT)
    _sh("git", "checkout", "--quiet", COMMIT)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "."], check=True)
ROOT = os.getcwd()
print(ROOT)
print(_sh("git", "log", "-1", "--oneline"))

/workspace/target_repo
627b015 Remyx: propose .remyx/validation.yaml for this change


## 3. Credentials

If the benchmark downloads gated models or datasets it needs a Hugging Face token. In Colab, store it as a secret named `HF_TOKEN`; elsewhere set the environment variable.

In [5]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
print("HF_TOKEN set" if os.environ.get("HF_TOKEN") else "HF_TOKEN not set — gated downloads will fail")

HF_TOKEN set


## 4. The experiment configuration

The harness runs a method by its configuration directory. This validation points it at `experiments/lora/llama-3.2-3B-rank32-kappa05` (relative to `method_comparison/MetaMathQA`).

`method_comparison/MetaMathQA/experiments/lora/llama-3.2-3B-rank32-kappa05/adapter_config.json`:

```json
{
  "peft_type": "LORA",
  "base_model_name_or_path": "meta-llama/Llama-3.2-3B",
  "r": 32,
  "lora_alpha": 64,
  "lora_dropout": 0.0,
  "target_modules": ["v_proj", "q_proj"],
  "condition_number_top_fraction": 0.5
}
```

In [6]:
print(open(os.path.join(ROOT, "method_comparison/MetaMathQA/experiments/lora/llama-3.2-3B-rank32-kappa05/adapter_config.json")).read())

{
  "peft_type": "LORA",
  "base_model_name_or_path": "meta-llama/Llama-3.2-3B",
  "r": 32,
  "lora_alpha": 64,
  "lora_dropout": 0.0,
  "target_modules": ["v_proj", "q_proj"],
  "condition_number_top_fraction": 0.5
}


## 5. Confirm the change under test is what is loaded

The commit printed here must match the one checked out above.

In [7]:
import importlib
print(_sh("git", "rev-parse", "HEAD"))

627b015c7fd1ae77e5110980be98217760f6479f


## 6. Run the benchmark

`method_comparison/MetaMathQA/run.py` over `experiments/lora/llama-3.2-3B-rank32-kappa05` — a directory of experiments runs each in turn; a single experiment runs once.

In [8]:
os.chdir(os.path.join(ROOT, "method_comparison/MetaMathQA"))
import glob, importlib, runpy, sys, time
RUN_STARTED = time.time()
configs = sorted(glob.glob("experiments/lora/llama-3.2-3B-rank32-kappa05/*/")) or ["experiments/lora/llama-3.2-3B-rank32-kappa05"]
for cfg in configs:
    print(f"[remyx] {cfg}")
    sys.argv = ["run.py", cfg.rstrip("/")]
    runpy.run_path("run.py", run_name="__main__")

[remyx] experiments/lora/llama-3.2-3B-rank32-kappa05


/root/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 2 files:  50%|█████     | 1/2 [00:10<00:10, 10.06s/it]

Fetching 2 files: 100%|██████████| 2/2 [00:22<00:00, 11.18s/it]

...

 93%|█████████▎| 25/27 [11:12<00:52, 26.29s/it]

 96%|█████████▋| 26/27 [11:34<00:25, 25.02s/it]

100%|██████████| 27/27 [11:43<00:00, 20.37s/it]

100%|██████████| 27/27 [11:43<00:00, 26.07s/it]

accelerator memory max: 20216MB
accelerator memory reserved avg: 13236MB
accelerator memory reserved 99th percentile: 18342MB
train time: 2764.0175771370023s
total time: 3411.46s
file size of checkpoint: 21.0MB


## 7. Read what the benchmark wrote

Results land under `temporary_results/lora--llama-3.2-3B-rank32-kappa05--*.json` (relative to `method_comparison/MetaMathQA`) — or wherever this harness writes for a non-default checkout; only a document written by the run above counts. The metrics the criteria are judged against are fields of that document.

In [9]:
import glob, json, os
PATTERNS = ["temporary_results/lora--llama-3.2-3B-rank32-kappa05--*.json"]
paths = sorted((p for pat in PATTERNS for p in glob.glob(pat)), key=os.path.getmtime)
paths = [p for p in paths if os.path.getmtime(p) >= RUN_STARTED - 1]
if not paths:
    # Some harnesses write elsewhere depending on the checkout (peft uses
    # temporary_results/ off the main branch): any document this run wrote.
    paths = sorted((p for p in glob.glob("**/*.json", recursive=True)
                    if os.path.getmtime(p) >= RUN_STARTED - 1 and "experiments/" not in p),
                   key=os.path.getmtime)
assert paths, "the benchmark wrote no result document"
doc = json.load(open(paths[-1]))
print("result document:", paths[-1])

def find(obj, key):
    """Last value under `key` anywhere in the document ('test accuracy' matches test_accuracy)."""
    hit = None
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).replace(" ", "_") == key and isinstance(v, (int, float)):
                hit = v
            found = find(v, key)
            hit = found if found is not None else hit
    elif isinstance(obj, list):
        for item in obj:
            found = find(item, key)
            hit = found if found is not None else hit
    return hit

METRICS = ["num_trainable_params", "test_accuracy", "forgetting", "total_time", "train_time", "accelerator_memory_max"]
observed = {name: find(doc, name) for name in METRICS}
print(json.dumps(observed, indent=2))

result document: temporary_results/lora--llama-3.2-3B-rank32-kappa05--2026-09-13T00-06-12+00-00.json
{
  "num_trainable_params": 5505024,
  "test_accuracy": 0.42987111448066717,
  "forgetting": 0.024042248725891113,
  "total_time": 3411.463763422,
  "train_time": 92.77018990900615,
  "accelerator_memory_max": 21198012416
}


## 8. Against the criteria

Thresholds come from `.remyx/validation.yaml`, so a failing measurement reports rather than crashes. `baseline` is the published row this repository already ships for the comparison method.

In [10]:
CRITERIA = [
    {
        "metric": "num_trainable_params",
        "direction": "<=",
        "threshold": 5505024,
        "baseline": 9175040.0
    },
    {
        "metric": "test_accuracy",
        "direction": ">=",
        "threshold": 0.4705,
        "baseline": 0.49052312357846856
    },
    {
        "metric": "forgetting",
        "direction": "<=",
        "threshold": 0.4357,
        "baseline": 0.4156990051269531
    },
    {
        "metric": "total_time",
        "direction": "<=",
        "threshold": 1173.714544863964,
        "baseline": 1173.714544863964
    },
    {
        "metric": "train_time",
        "direction": "<=",
        "threshold": 958.3276851720293,
        "baseline": 958.3276851720293
    },
    {
        "metric": "accelerator_memory_max",
        "direction": "<=",
        "threshold": 22286434304.0,
        "baseline": 22286434304.0
    }
]

print(f"{'metric':<28}{'observed':>16}{'baseline':>16}  criterion")
for c in CRITERIA:
    v = observed.get(c["metric"])
    t = c["threshold"]
    ok = None if v is None or t is None else (v <= t if c["direction"] == "<=" else v >= t)
    mark = "?" if ok is None else ("PASS" if ok else "FAIL")
    fmt = lambda x: (f"{x:.6g}" if isinstance(x, float) else str(x))
    print(f"{c['metric']:<28}{fmt(v):>16}{fmt(c['baseline']):>16}  {c['direction']} {fmt(t)}  {mark}")

metric                              observed        baseline  criterion
num_trainable_params                 5505024     9.17504e+06  <= 5505024  PASS
test_accuracy                       0.429871        0.490523  >= 0.4705  FAIL
forgetting                         0.0240422        0.415699  <= 0.4357  PASS
total_time                           3411.46         1173.71  <= 1173.71  FAIL
train_time                           92.7702         958.328  <= 958.328  PASS
accelerator_memory_max           21198012416     2.22864e+10  <= 2.22864e+10  PASS


## 9. Report

One line, machine-readable — what Remyx records as this run's measurement.

In [11]:
print(json.dumps(observed))

{"num_trainable_params": 5505024, "test_accuracy": 0.42987111448066717, "forgetting": 0.024042248725891113, "total_time": 3411.463763422, "train_time": 92.77018990900615, "accelerator_memory_max": 21198012416}


## 10. What the outcome means

- **All rows pass** → the claim holds at this protocol: `num_trainable_params` <= 5505024 with `test_accuracy` >= 0.4705, `forgetting` <= 0.4357, `total_time` <= 1173.714544863964, `train_time` <= 958.3276851720293, `accelerator_memory_max` <= 22286434304.0 holding.
- **`num_trainable_params` fails** → the change does not deliver what the claim says at this protocol.
- **A guardrail fails** → the target may be met at the cost of something the claim promised to keep; look at the run log before drawing a conclusion.
- **No result document** → the benchmark did not finish; the run cell above says why.

## Appendix — the criteria file

`.remyx/validation.yaml` as committed:

```yaml
model:
  provider: zai
loop:
  max_iterations: 8
  fix_code: true

benchmarks:
  - name: kappa-lora-spectral-targeting
    suite:
      harness:
        notebook: .remyx/validations/kappa-lora-spectral-targeting.ipynb
        runner: method_comparison/MetaMathQA/run.py
        experiments: experiments/lora/llama-3.2-3B-rank32-kappa05
        results_glob: "temporary_results/lora--llama-3.2-3B-rank32-kappa05--*.json"
        method: lora
        smoke:
          params_path: method_comparison/MetaMathQA/default_training_params.json
          # truncate the published 5000-step regime to a few minutes of plumbing proof;
          # keys must be the step-count / eval-interval / generation-length keys of that file
          overrides:
            max_steps: 20
            eval_steps: 10
      scorer: num_trainable_params
      metrics:
        - name: num_trainable_params
          role: target
          direction: min
          # derivation: pool = 28 q_proj (r*(3072+3072)=196608 each) + 28 v_proj (r*(3072+1024)=131072 each)
          # = 9175040, reproducing the published row exactly. top-50% keeps ceil(56*0.5)=28 modules;
          # all-q worst case 28*196608 = 5505024 (60.0%), all-v floor 3670016 (40.0%), 14/14 split 4587520 (exactly 50%).
          # above 5505024 implies >28 modules injected = selection bug; expect ~4.6M.
          threshold: 5505024
        - name: test_accuracy
          role: guardrail
          direction: max
          # published row 0.49052312357846856 minus the ±0.02 parity band this repo's own harness notebook uses
          threshold: 0.4705
        - name: forgetting
          role: cost
          direction: min
          # published 0.4156990051269531 + 0.02 band — reporting reference only, never a gate (user guidance:
          # a noisy forgetting number must not veto the param-count/accuracy question)
          threshold: 0.4357
        - name: total_time
          role: cost
          direction: min
          # no-regression vs the published row; cost reference, not a parity target (paper's -16.2% is suite-averaged)
          threshold: 1173.714544863964
        - name: train_time
          role: cost
          direction: min
          threshold: 958.3276851720293
        - name: accelerator_memory_max
          role: cost
          direction: min
          # halving ~4.6M trainable params saves ~37MB optimizer state vs a 22.3GB peak — inside noise, report only
          threshold: 22286434304.0
    baseline:
      source: method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json
      values:
        num_trainable_params: 9175040.0
        test_accuracy: 0.49052312357846856
        forgetting: 0.4156990051269531
        total_time: 1173.714544863964
        train_time: 958.3276851720293
        accelerator_memory_max: 22286434304.0
    compute:
      tier: gpu
      # published row total_time=1173.7s (~20 min on A100 incl. GSM8K eval) + gated Llama-3.2-3B download and
      # MetaMathQA tokenization margin (~25 min) + headroom -> 5400s for ONE arm
      timeout_s: 5400
    policy:
      guardrail_veto: true

held_constant:
  - "base model meta-llama/Llama-3.2-3B exactly as the published lora/llama-3.2-3B-rank32 row (gated repo, HF_TOKEN provided)"
  - "LoRA hyperparameters copied verbatim from the row: r=32, lora_alpha=64, lora_dropout=0.0, target_modules=[v_proj, q_proj]"
  - "training/eval regime = the harness default_training_params.json (5000 steps MetaMathQA, GSM8K eval); no training_params.json override in the new experiment dir"
  - "same invocation as the requester's Colab pattern: python run.py --verbose --clean <experiment> from method_comparison/MetaMathQA"
  - "condition_number_top_fraction=0.5 is the ONLY delta vs the published row's config"
avoid:
  - "do not widen target_modules beyond q_proj/v_proj — the published row defines the 56-module candidate pool the threshold arithmetic assumes"
  - "unpinned revisions of the base model or datasets; keep base_model_name_or_path byte-identical to the row"
  - "treating wall-clock or memory deltas as parity targets — they are no-regression cost references only, since the LoRA share of the 22.3GB peak is tiny"
  - "routing the eval through add_weighted_adapter — the diff explicitly bypasses spectral targeting there"
  - "substituting a synthetic or CPU proxy for the MetaMathQA/GSM8K run the claim names"
provenance:
  num_trainable_params: "published_row:method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json"
  test_accuracy: "published_row:method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json"
  suite: "user_resource:https://colab.research.google.com/drive/1z73-jtAGrq4HkjvorjFcZ77uMwdmWs56?usp=sharing"
  experiments: "user_guidance"
  target_threshold: "inferred: 28*196608 all-q_proj worst case among ceil(56*0.5)=28 selected modules"
  accuracy_guardrail: "user_resource:colab parity band ±0.02 applied to the published 0.490523"
  forgetting_cost_role: "user_guidance: keep forgetting, time and memory as cost references only — must not veto the run"
  held_constant: "protocol_doc:method_comparison/MetaMathQA/README.md"
  compute: "inferred: the claim is about a 5000-step fine-tuning run; no CPU-valid instrument observes fit or param count under the published protocol"
```